In [1]:
import os

os.environ["KERAS_BACKEND"] = "jax"

import keras_hub

encoder = keras_hub.models.TextEmbedder.from_preset("all_minilm_l6_v2_en")

/Users/kotarohara/repo/teaching/cs702-ci/.pixi/envs/default/lib/python3.12/site-packages/keras/src/layers/layer.py:431: UserWarning: `build()` was called on layer 'bert_text_embedder', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


In [9]:
embedding = encoder.predict_on_batch(
    [
        "It's beautiful and sunny outside.",
        "The weather is very good today.",
        "It is raining.",
        "Oh wow, it is pouring",
    ]
)

print(embedding.shape)

(4, 384)


In [10]:
# Calculate similarities between the embedding vecotrs
import jax.numpy as jnp


def cosine_similarity(x, y):
    """Compute cosine similarity between two vectors."""
    x = x / (jnp.linalg.norm(x) + 1e-8)
    y = y / (jnp.linalg.norm(y) + 1e-8)
    return jnp.dot(x, y)


print(cosine_similarity(embedding[0], embedding[1]))
print(cosine_similarity(embedding[0], embedding[2]))
print(cosine_similarity(embedding[0], embedding[3]))


0.6161209
0.43634173
0.29094055


In [11]:
weather_examples = {
    "rainy": [
        "It is raining outside.",
        "The weather is rainy.",
        "It's pouring outside.",
    ],
    "sunny": [
        "It is sunny outside.",
        "The weather is bright and sunny.",
        "It's a beautiful sunny day.",
    ],
}

mood_examples = {
    "active": [
        "I feel energetic.",
        "I want to do something active.",
        "I have lots of energy.",
    ],
    "chill": [
        "I want to relax.",
        "I feel like taking it easy.",
        "I want to do something calm.",
    ],
}

In [12]:
def make_prototype(examples):
    """Create a prototype embedding from several examples."""
    embeddings = encoder.predict_on_batch(examples)
    prototype = jnp.mean(embeddings, axis=0)
    return prototype / (jnp.linalg.norm(prototype) + 1e-8)


weather_prototypes = {
    name: make_prototype(examples) for name, examples in weather_examples.items()
}

mood_prototypes = {
    name: make_prototype(examples) for name, examples in mood_examples.items()
}

In [13]:
def similarity_scores(embedding, prototypes):
    """Compare an embedding with each prototype."""
    return {
        name: float(cosine_similarity(embedding, prototype))
        for name, prototype in prototypes.items()
    }


utterance = "I'm full of energy today."
x = encoder.predict_on_batch([utterance])

print(similarity_scores(x[0], mood_prototypes))
# {'active': 0.7348935604095459, 'chill': 0.42395609617233276}

{'active': 0.7348935604095459, 'chill': 0.42395609617233276}


In [14]:
def match_prototype(embedding, prototypes, threshold=0.5):
    """Return the closest prototype if similarity is high enough."""
    scores = similarity_scores(embedding, prototypes)

    best_label = max(scores, key=scores.get)
    best_score = scores[best_label]

    if best_score < threshold:
        return None

    return best_label


match_prototype(x[0], mood_prototypes)
# 'active'

'active'

In [15]:
from frame import InputEvent  # Imports InputFrame from the neighboring file


def process_input(utterance, threshold=0.5):
    """Interpret natural-language input."""
    x = encoder.predict_on_batch([utterance])[0]

    weather = match_prototype(
        x,
        weather_prototypes,
        threshold,
    )

    mood = match_prototype(
        x,
        mood_prototypes,
        threshold,
    )

    return InputEvent(
        weather=weather,
        mood=mood,
    )


input_event = process_input("I'm full of energy today.")
print(input_event)
# InputEvent(weather=None, mood='active')

InputEvent(weather=None, mood='active')
